# cellmap-flow on Colab (full demo: backend + frontend)

Run cellmap-flow's inference server **and** the browser dashboard inside this
Colab session, exposed via two public Cloudflare tunnels. Free Colab gives
you a T4 GPU — fast inference + the full dashboard pipeline UI.

The session ends when this Colab tab closes; tunnels die with it.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run all cells.
3. Open the printed `https://*.trycloudflare.com/dashboard.html?...` URL.
4. In the dashboard, click **Open in NG**. Pan around; inference runs on
   the Colab GPU.

## 1. Install everything

cellmap-flow (with the `bioimageio` extra so BMZ models work too), Node.js
(to build the frontend), cloudflared, and a checkout of the repo.

Takes ~3–5 minutes the first time; cached on re-run.

In [ ]:
# cellmap-flow with the bioimageio extra (so both `huggingface` and
# `bioimage` server subcommands work). Pinned to the branch with the
# https:// + anonymous-S3 + BMZ fixes.
%pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs

# Node.js for building the browser frontend.
!command -v node >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq nodejs npm)
!node --version && npm --version

# cloudflared static binary for the public tunnels.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

# Repo checkout for the browser/ directory.
import os, subprocess
REPO_DIR = "/content/cellmap-flow-repo"
if not os.path.isdir(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth=1", "--branch=browser-inference",
        "https://github.com/janelia-cellmap/cellmap-flow.git", REPO_DIR,
    ])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--depth=1", "origin", "browser-inference"])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", "origin/browser-inference"])
print("repo at:", REPO_DIR)


## 2. Configure model + dataset

Pick one of two model types:
- `huggingface` — a cellmap HF model (e.g. `cellmap/jrc_mus-livers_16nm_to_8nm_mito`).
- `bioimage` — a BioImage Model Zoo model (e.g. `hiding-blowfish`, 2D mito EM).

Set the right block below for your choice. Set `MODEL_TYPE` to switch.

> Heads up: cellmap HF UNets have very large `inference_input_shape` —
> they're sized for workstation GPUs and may OOM on Colab T4 (16 GB).
> Use a small dataset or smaller-read-shape model if you hit it.

In [ ]:
MODEL_TYPE = "huggingface"   # or "bioimage"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"  # 178^3 inference; fits T4
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)
# Other T4-friendly options:
#   HF_REPO = "cellmap/fly_organelles_run07_700000"     # 178^3, fly organelles, different ckpt
#   HF_REPO = "cellmap/fly_organelles_run08_438000"     # 178^3, fly organelles, run08
#   HF_REPO = "cellmap/jrc_mus-livers_16nm_to_8nm_mito" # 288^3, BORDERLINE on T4 (OOM-prone)


# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"      # nm per voxel
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

# Local ports used inside Colab; cloudflared maps each to a public URL.
BACKEND_PORT = 8765
FRONTEND_PORT = 4173

# Derived: pick which block actually drives the rest of the notebook.
if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")


## 3. Start the cellmap-flow inference server

Runs in the background so the frontend + tunnels can come up alongside.

In [ ]:
import os, subprocess, time

# Try to reduce CUDA fragmentation. cellmap-flow's UNets are big enough
# that on free Colab T4 (16 GB) you can hit OOM partway through forward.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO,
        "--name", HF_NAME,
        "-d", HF_DATASET,
        "--port", str(BACKEND_PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL,
        "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET,
        "--port", str(BACKEND_PORT),
    ]

print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{BACKEND_PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{BACKEND_PORT}" in line:
        print("\n[server] ready.")
        break


## 4. Public cloudflared tunnel for the inference server

In [ ]:
import subprocess, re, time

tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{BACKEND_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
BACKEND_URL = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        BACKEND_URL = m.group(0)
        break

print("\n" + "=" * 70)
print(f"BACKEND URL: {BACKEND_URL}")
print("=" * 70)


## 5. Build the cellmap-flow browser frontend

Compiles `cellmap-flow-repo/browser/` to a static bundle. Takes ~1–2 minutes
on first run; cached on re-runs.

In [ ]:
import subprocess, os
os.chdir(f"{REPO_DIR}/browser")
subprocess.check_call(["npm", "install", "--no-audit", "--no-fund", "--no-progress"])
subprocess.check_call(["npm", "run", "build"])
os.chdir("/content")
print("frontend built at:", f"{REPO_DIR}/browser/dist")


## 6. Serve the frontend on a local port

Plain `python -m http.server` against the prebuilt `browser/dist/`.
Cloudflared tunnels this port out next.

In [ ]:
import subprocess, time, socket

# Drop any stale http.server that's bound to this port (re-runs).
for p in subprocess.run(["pgrep", "-f", "http.server.*--directory.*browser/dist"],
                        capture_output=True, text=True).stdout.split():
    subprocess.run(["kill", p])
time.sleep(0.5)

frontend = subprocess.Popen(
    ["python3", "-m", "http.server", str(FRONTEND_PORT),
     "--directory", f"{REPO_DIR}/browser/dist"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
# Wait for the port to actually open.
for _ in range(40):
    try:
        with socket.create_connection(("127.0.0.1", FRONTEND_PORT), timeout=0.5):
            print(f"frontend listening on :{FRONTEND_PORT}")
            break
    except OSError:
        time.sleep(0.25)
else:
    raise RuntimeError("frontend http.server didn't bind in time")


## 7. Cloudflared tunnel for the frontend

In [ ]:
import subprocess, re, time

tunnel_fe = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{FRONTEND_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
FRONTEND_URL = None
for _ in range(120):
    line = tunnel_fe.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        FRONTEND_URL = m.group(0)
        break

print("\n" + "=" * 70)
print(f"FRONTEND URL: {FRONTEND_URL}")
print("=" * 70)


## 8. One-click demo link

Opens the dashboard with the backend URL + dataset pre-filled. Click
**Open in NG** once the page loads.

In [ ]:
import urllib.parse

if not (BACKEND_URL and FRONTEND_URL):
    print("missing BACKEND_URL or FRONTEND_URL; check the cloudflared cells above.")
else:
    qp = {
        "backend": BACKEND_URL,
        # Dataset slug for the server-backed flow: cellmap-flow's server
        # treats this as a per-request key and serves the configured zarr.
        "dataset": MODEL_NAME,
        # ?raw= adds the source EM zarr as a second NG layer so the user
        # sees what the model is segmenting alongside the prediction.
        "raw": DATASET,
    }
    if MODEL_TYPE == "huggingface":
        qp["hf"] = HF_REPO

    qs = urllib.parse.urlencode(qp, safe=":/")
    DEMO_URL = f"{FRONTEND_URL}/dashboard.html?{qs}"

    print("*" * 70)
    print("DEMO URL (open in any browser):")
    print(DEMO_URL)
    print("*" * 70)
    print()
    print("Direct cellmap-flow inference server (for pasting into other tools):")
    print(f"  {BACKEND_URL}")


## 9. Keep the runtime alive

Run this cell last so Colab doesn't idle-disconnect. Stop it with the ▢
button to tear everything down.

In [ ]:
import time, select

procs = [(server, "server"), (tunnel, "tunnel-backend"),
         (frontend, "frontend"), (tunnel_fe, "tunnel-frontend")]

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r:
            return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        died = False
        for p, label in procs:
            drain(p, label)
            if p.poll() is not None:
                drain(p, label)
                print(f"\n[{label}] exited rc={p.returncode}. logs above.")
                died = True
        if died: break
        time.sleep(2)
finally:
    for p, _ in procs:
        try: p.terminate()
        except Exception: pass
